<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/Module4_Labs/Lab14.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 14 — Optional: VQE for a Small Materials-Model Hamiltonian
**Quantum Optimization and Simulation — VQE Laboratory Series**

This optional lab shows that VQE is not only a chemistry method. We use a small transverse-field Ising model from condensed-matter/materials physics:

\[
H=-J(Z_0Z_1+Z_1Z_2)-h(X_0+X_1+X_2).
\]

The model is deliberately small and simplified. Its purpose is to reuse the same VQE workflow in a different application area.

**Suggested use:** optional enrichment or instructor demonstration.


## Learning objectives
- Apply the VQE workflow to a non-molecular Hamiltonian.
- Interpret the competition between the \(-JZZ\) interaction and the \(-hX\) field.
- Compare a variational answer with exact diagonalization.
- Observe how the lowest-energy state changes as the field strength changes.


In [ ]:
# Run once in a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp

SEED = 123


In [ ]:
from scipy.optimize import minimize
from qiskit.circuit import ParameterVector

## Part A — Build the Hamiltonian

In [ ]:
def ising_hamiltonian(J, h):
    return SparsePauliOp.from_list([
        ("IZZ", -J),
        ("ZZI", -J),
        ("IIX", -h),
        ("IXI", -h),
        ("XII", -h),
    ])

J = 1.0
h = 0.8
H = ising_hamiltonian(J, h)

print(H)
exact = np.linalg.eigvalsh(H.to_matrix())[0]
print("Exact ground-state energy:", exact)

## Part B — Hardware-efficient ansatz

In [ ]:
def materials_ansatz(params):
    qc = QuantumCircuit(3)

    # Layer 1: local rotations
    for q in range(3):
        qc.ry(params[q], q)

    # Entangling chain
    qc.cx(0,1)
    qc.cx(1,2)

    # Layer 2: more local rotations
    for q in range(3):
        qc.ry(params[3+q], q)

    return qc

## Part C — Variational energy and COBYLA optimization

For this optional lab, we use an exact statevector expectation value so that the new materials-physics idea stays the focus.


In [ ]:
history = []
objective_evaluations = 0

def exact_ansatz_energy(params, H):
    global objective_evaluations
    qc = materials_ansatz(params)
    psi = Statevector.from_instruction(qc)
    value = np.real(psi.expectation_value(H))
    history.append(value)
    objective_evaluations += 1
    return value

initial = np.zeros(6)

result = minimize(
    lambda p: exact_ansatz_energy(p, H),
    x0=initial,
    method="COBYLA",
    options={"maxiter":120, "rhobeg":0.5}
)

print("VQE energy:", result.fun)
print("Exact energy:", exact)
print("Error:", result.fun-exact)
print("Variational energy evaluations:", objective_evaluations)

plt.plot(history)
plt.xlabel("Objective evaluation")
plt.ylabel("Energy")
plt.title("Materials-model VQE convergence")
plt.show()


## Part D — Sweep the field strength

In [ ]:
field_values = np.linspace(0.0, 2.0, 9)
exact_energies = []
vqe_energies = []

for h in field_values:
    Hh = ising_hamiltonian(J=1.0, h=h)
    exact_energies.append(np.linalg.eigvalsh(Hh.to_matrix())[0])

    result_h = minimize(
        lambda p: exact_ansatz_energy(p, Hh),
        x0=np.zeros(6),
        method="COBYLA",
        options={"maxiter":80}
    )
    vqe_energies.append(result_h.fun)

plt.plot(field_values, exact_energies, "o-", label="Exact")
plt.plot(field_values, vqe_energies, "s--", label="VQE")
plt.xlabel("Transverse field h/J")
plt.ylabel("Ground-state energy")
plt.legend()
plt.show()

### YOUR TURN
For `h=0`, inspect the exact ground-state eigenvectors. Which computational-basis configurations dominate? Then compare with a large field, such as `h=2`.

## Reflection
1. What physical tendency is represented by the \(-JZZ\) terms?
2. What competing tendency is represented by the \(-hX\) terms?
3. Why is this lab relevant to materials modeling even though it is only a three-qubit toy model?

<details>
<summary><b>Instructor solution / suggested answer</b></summary>


    1. For positive \(J\), the \(-JZZ\) terms favor neighboring spins aligned in the Z direction.
    2. The transverse field favors alignment along the X direction, which competes with definite Z alignment and creates superposition.
    3. Materials physics frequently studies simplified effective Hamiltonians to isolate collective mechanisms. The same VQE workflow—prepare a trial state, measure energy terms, and optimize parameters—extends to larger and more realistic models.

</details>